# mem0 Drop-In Wrapper: Learn Playbooks From Your Existing mem0 Traffic

> **Time:** ~10 minutes | **Level:** Beginner

If your agent already uses [mem0](https://mem0.ai)'s managed platform, `reflexio.mem0` lets
Reflexio learn playbooks and profiles from the same traffic — **by changing one import**.

In this notebook you'll:
1. **Switch** the import and keep every mem0 call site unchanged
2. **Add** a conversation — stored in mem0 *and* published to Reflexio
3. **Search** — default search stays exactly mem0; opt in to one Reflexio namespace
4. **Verify** both sides end to end, including graceful degradation

### Prerequisites
- Reflexio server running (`uv run reflexio services start --only backend`)
- `MEM0_API_KEY` set in your environment (a real mem0 platform key)
- The mem0 extra installed: `pip install 'reflexio-ai[mem0]'` (or `uv run --with mem0ai ...`)

In [ ]:
import os
import time
import uuid

from _display_helpers import *

# Each run uses a unique ID so the notebook is idempotent
RUN_ID = uuid.uuid4().hex[:8]
USER_ID = f"mem0_demo_customer_{RUN_ID}"
AGENT_ID = f"support-agent-v1-{RUN_ID}"

load_env()
assert os.environ.get("MEM0_API_KEY"), "Set MEM0_API_KEY before running this notebook"
show_success(f"mem0 key present, test user: {USER_ID}, test agent: {AGENT_ID}")

## The One-Line Migration

```python
# Before
from mem0 import MemoryClient
# After
from reflexio.mem0 import MemoryClient
```

The wrapped client **is** a `mem0.MemoryClient` (a subclass), so every method, attribute,
and `isinstance` check behaves exactly as before. Reflexio credentials come from
`REFLEXIO_URL` / `REFLEXIO_API_KEY` env vars, or an explicit `reflexio_client=`.

In [ ]:
import mem0

from reflexio.mem0 import MemoryClient

client = MemoryClient(api_key=os.environ["MEM0_API_KEY"])

assert isinstance(client, mem0.MemoryClient)
assert client.reflexio.configured, "Set REFLEXIO_URL or REFLEXIO_API_KEY"
show_success("Wrapped client constructed — still a genuine mem0.MemoryClient")

## Add a Conversation

`add()` behaves exactly like mem0's: memories are extracted and stored in your mem0 account,
and the call returns mem0's own payload. In addition, the wrapper publishes the conversation
to Reflexio (best-effort, `wait_for_response=False`) so it can learn playbooks:

| mem0 argument | Reflexio field |
|---|---|
| `user_id` | `user_id` (**required** for the publish) |
| `run_id` | `session_id` |
| `agent_id` | `agent_version` |

We publish **8 turns**: Reflexio's default extraction gate (`stride_size=8`) waits for
enough new interactions before running profile/playbook extraction, so short test
conversations below that threshold won't produce profiles.


In [ ]:
conversation = [
    {"role": "user", "content": "Hi, I ordered the espresso machine last week and it arrived with a cracked water tank. I travel Mondays to Wednesdays, so any replacement has to be delivered Thursday or Friday."},
    {"role": "assistant", "content": "Sorry about the damage! I've arranged a replacement with Thursday delivery and added a note that you're only available Thursdays and Fridays."},
    {"role": "user", "content": "Great. Also, please always email me the invoice as a PDF attachment — I can't open the web links from my work laptop."},
    {"role": "assistant", "content": "Done — invoices for your account will be sent as PDF attachments from now on."},
    {"role": "user", "content": "One more thing: I'm lactose intolerant, so when you suggest coffee recipes or accessories, skip anything dairy-based — oat milk works for me."},
    {"role": "assistant", "content": "Noted! I'll only recommend dairy-free options like oat milk for recipes and steaming accessories."},
    {"role": "user", "content": "And can you address me as Sam rather than my full name? Also I'm in the Pacific time zone, so please don't schedule calls before 9am PT."},
    {"role": "assistant", "content": "Of course, Sam! I've noted your Pacific time zone and will keep any calls after 9am PT."},
]

mem0_result = client.add(
    conversation,
    user_id=USER_ID,
    run_id=f"support-session-{RUN_ID}",
    agent_id=AGENT_ID,
)
show_json(mem0_result, title="mem0 add() payload (unchanged by the wrapper)")

## Verify Side 1: Memories Landed in mem0

Everything below uses plain mem0 API calls — the wrapper forwards them verbatim.
mem0 extracts memories asynchronously, so we poll briefly.

In [ ]:
mem0_memories = []
for _ in range(20):
    page = client.get_all(filters={"user_id": USER_ID})
    mem0_memories = page.get("results", [])
    if mem0_memories:
        break
    time.sleep(3)

assert mem0_memories, "mem0 returned no memories for the test user"
show_json([m.get("memory") for m in mem0_memories], title=f"{len(mem0_memories)} memories in mem0")

## Default Search Is Still Exactly mem0

Without `include_reflexio=True`, search does not inspect or call Reflexio and adds no keys.
Your existing mem0 search call therefore keeps its original result schema.

In [ ]:
plain_result = client.search(
    "How should this customer's deliveries and invoices be handled?",
    filters={"user_id": USER_ID, "agent_id": AGENT_ID},
)
assert "results" in plain_result and "reflexio" not in plain_result
show_json(plain_result["results"], title="Unchanged default mem0 search")

## Search: mem0 Results + Reflexio Learnings in One Call

Set `include_reflexio=True` to receive a shallow copy with one `reflexio` object.
The wrapper never inserts this content into a prompt. Your application chooses how to
validate and format both result sets; treat all retrieved text as untrusted input.

Reflexio extracts profiles/playbooks asynchronously after the publish, so we poll until
at least one real learned artifact is available. The notebook fails if none appears.

In [ ]:
deadline = time.time() + 180
search_result = None
while time.time() < deadline:
    search_result = client.search(
        "How should this customer's deliveries and invoices be handled?",
        filters={"user_id": USER_ID, "agent_id": AGENT_ID},
        include_reflexio=True,
    )
    learnings = search_result["reflexio"]
    assert learnings["status"] == "ok", learnings
    if any(learnings[key] for key in ("profiles", "user_playbooks", "agent_playbooks")):
        break
    time.sleep(10)

assert search_result is not None
learnings = search_result["reflexio"]
assert any(learnings[key] for key in ("profiles", "user_playbooks", "agent_playbooks")), (
    "Reflexio did not produce a learned artifact before the deadline"
)

# Keep the complete, agent-scoped records so cleanup can target only IDs
# created for this run.
agent_playbook_ids = sorted(
    {
        playbook["agent_playbook_id"]
        for playbook in learnings["agent_playbooks"]
        if playbook.get("agent_playbook_id")
    }
)

show_json([m.get("memory") for m in search_result.get("results", [])], title="mem0 search results")
show_json(
    {
        "profiles": [p["content"] for p in learnings["profiles"]],
        "user_playbooks": [p["content"] for p in learnings["user_playbooks"]],
        "agent_playbooks": [p["content"] for p in learnings["agent_playbooks"]],
    },
    title="Opt-in Reflexio namespace",
)
show_success("Reflexio learned from the mem0 traffic and returned a namespaced artifact")

## Graceful Degradation

If Reflexio is down or misconfigured, the wrapper must never break your mem0 code path:
`add()` still stores memories and default `search()` stays exact. Opted-in search reports
a safe `error` envelope instead of raising or leaking transport details.

In [ ]:
from reflexio import ReflexioClient

degraded = MemoryClient(
    api_key=os.environ["MEM0_API_KEY"],
    reflexio_client=ReflexioClient(url_endpoint="http://127.0.0.1:1", timeout=2),
)
filters = {"user_id": USER_ID, "agent_id": AGENT_ID}
r = degraded.add("I prefer morning deliveries.", **filters, run_id=f"degraded-{RUN_ID}")
s = degraded.search("deliveries", filters=filters)
enriched = degraded.search("deliveries", filters=filters, include_reflexio=True)
# Managed mem0 add() returns an async event envelope; search() returns {"results": [...]}
assert isinstance(r, dict) and r
assert "results" in s
assert "reflexio" not in s
assert enriched["reflexio"]["status"] == "error"
assert enriched["reflexio"]["reason"] == "request_failed"
show_success("Reflexio unreachable → mem0 stays intact and opt-in status is explicit")

## Cleanup

Inherited methods like `delete_all` affect mem0 only. Mirrored data is deleted
explicitly through the scope-aware `client.reflexio` facade; organization-wide
deletion is not exposed. Already-derived agent playbooks require exact IDs.

In [ ]:
client.delete_all(user_id=USER_ID, agent_id=AGENT_ID)
# mem0 deletion is asynchronous server-side; poll briefly.
remaining = client.get_all(
    filters={"user_id": USER_ID, "agent_id": AGENT_ID}
).get("results", [])
for _ in range(10):
    if not remaining:
        break
    time.sleep(3)
    remaining = client.get_all(
        filters={"user_id": USER_ID, "agent_id": AGENT_ID}
    ).get("results", [])

session_cleanup = client.reflexio.delete_session_records(
    user_id=USER_ID, agent_id=AGENT_ID, run_id=f"support-session-{RUN_ID}"
)
user_cleanup = client.reflexio.clear_user_data(user_id=USER_ID)
agent_cleanup = (
    client.reflexio.delete_agent_playbooks_by_ids(agent_playbook_ids)
    if agent_playbook_ids
    else None
)

assert not remaining, f"mem0 still exposes {len(remaining)} memories after cleanup"
assert session_cleanup.success
assert user_cleanup.success, user_cleanup.message
assert agent_cleanup is None or agent_cleanup.success, agent_cleanup.message
show_success(
    f"Cleaned this run from mem0 and Reflexio "
    f"({len(agent_playbook_ids)} agent playbook(s) deleted)"
)


## What This Validated

- **Drop-in**: one import change; the wrapped client is a genuine `mem0.MemoryClient`
- **Dual write**: one `add()` stored memories in mem0 *and* published the trace to Reflexio
- **Exact default read**: normal `search()` kept mem0's schema and made no Reflexio request
- **Opt-in read**: `include_reflexio=True` returned one explicit status namespace
- **No prompt injection**: the caller remains responsible for formatting retrieved context
- **Lifecycle**: mem0 deletion stayed unchanged and Reflexio cleanup used `client.reflexio`
- **Safety**: a dead Reflexio endpoint never breaks the mem0 path

Next: see [03_playbook.ipynb](03_playbook.ipynb) for how Reflexio aggregates and governs
the playbooks it learns from this traffic.